set up environment

In [ ]:
!pip install ultralytics
from ultralytics import YOLO
import os
import zipfile
import shutil
import random

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


data extraction and splitting

In [ ]:
import os
import random
import shutil
from collections import defaultdict

# --- CONFIGURATION ---
SOURCE_DIR = "/content/drive/MyDrive/images"
OUTPUT_DIR = "/content/dataset"
SPLIT_RATIO = 0.8
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

# Create destination directories
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'labels'), exist_ok=True)

all_files = os.listdir(SOURCE_DIR)

# 1. Map out every image-label pair and count global class distributions
dataset_pairs = []
global_class_counts = defaultdict(int)
image_map = {os.path.splitext(f)[0]: f for f in all_files if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS}

print("Analyzing dataset and counting all class indices...")

for stem, img_file in image_map.items():
    lbl_file = f"{stem}.txt"
    if lbl_file in all_files:
        lbl_path = os.path.join(SOURCE_DIR, lbl_file)

        # Track which classes are present in this specific file
        file_classes = []
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    class_idx = int(parts[0]) # Get the class index
                    file_classes.append(class_idx)
                    global_class_counts[class_idx] += 1

        dataset_pairs.append({
            'img': img_file,
            'lbl': lbl_file,
            'classes': file_classes
        })

print(f"Found {len(dataset_pairs)} total valid pairs across {len(global_class_counts)} unique classes.")

# 2. Sort pairs by rarity (images with the rarest classes are allocated first to prevent imbalance)
# This acts as a greedy stratification algorithm
def get_rarest_class_score(pair):
    if not pair['classes']:
        return float('inf') # Push unlabeled/empty images to the end
    return min(global_class_counts[c] for c in pair['classes'])

random.seed(42)
random.shuffle(dataset_pairs) # Randomize ties
dataset_pairs.sort(key=get_rarest_class_score)

# 3. Stratify and allocate to Train/Val while keeping track of current counts
train_pairs = []
val_pairs = []

train_class_counts = defaultdict(int)
val_class_counts = defaultdict(int)

for pair in dataset_pairs:
    # Evaluate which split needs the classes in this image more
    train_score = 0
    val_score = 0

    for c in pair['classes']:
        total_needed_train = global_class_counts[c] * SPLIT_RATIO
        total_needed_val = global_class_counts[c] * (1 - SPLIT_RATIO)

        # Calculate how far behind the ideal target each split currently is
        if train_class_counts[c] < total_needed_train:
            train_score += (total_needed_train - train_class_counts[c]) / total_needed_train
        if val_class_counts[c] < total_needed_val:
            val_score += (total_needed_val - val_class_counts[c]) / total_needed_val

    # Default to 80/20 fallback if scores are tied or image is empty
    if train_score == val_score:
        if len(train_pairs) < len(dataset_pairs) * SPLIT_RATIO:
            train_score += 1
        else:
            val_score += 1

    # Assign to the split that has the higher deficit score
    if train_score >= val_score:
        train_pairs.append((pair['img'], pair['lbl']))
        for c in pair['classes']:
            train_class_counts[c] += 1
    else:
        val_pairs.append((pair['img'], pair['lbl']))
        for c in pair['classes']:
            val_class_counts[c] += 1

# 4. Helper function to copy files
def copy_pairs(pairs, split_name):
    for img_file, lbl_file in pairs:
        shutil.copy(os.path.join(SOURCE_DIR, img_file), os.path.join(OUTPUT_DIR, split_name, 'images', img_file))
        shutil.copy(os.path.join(SOURCE_DIR, lbl_file), os.path.join(OUTPUT_DIR, split_name, 'labels', lbl_file))

print("Copying balanced files into Train and Validation directories...")
copy_pairs(train_pairs, 'train')
copy_pairs(val_pairs, 'val')

# 5. Print out the final balanced report
print("\n" + "="*50)
print(" STALWART MULTI-CLASS SPLIT COMPLETE REPORT ")
print("="*50)
print(f"Total Images -> Train: {len(train_pairs)} | Val: {len(val_pairs)}\n")
print(f"{'Class ID':<10} | {'Total Count':<12} | {'Train Count (80%)':<18} | {'Val Count (20%)':<15} | {'Actual Split Ratio':<18}")
print("-" * 85)

for c in sorted(global_class_counts.keys()):
    t_cnt = train_class_counts[c]
    v_cnt = val_class_counts[c]
    total = global_class_counts[c]
    ratio = (t_cnt / total) * 100 if total > 0 else 0
    print(f"{c:<10} | {total:<12} | {t_cnt:<18} | {v_cnt:<15} | {ratio:.1f}% / {100-ratio:.1f}%")
print("="*85)

Analyzing dataset and counting all class indices...
Found 1375 total valid pairs across 8 unique classes.
Copying balanced files into Train and Validation directories...

 STALWART MULTI-CLASS SPLIT COMPLETE REPORT 
Total Images -> Train: 1100 | Val: 275

Class ID   | Total Count  | Train Count (80%)  | Val Count (20%) | Actual Split Ratio
-------------------------------------------------------------------------------------
0          | 82           | 65                 | 17              | 79.3% / 20.7%
1          | 89           | 71                 | 18              | 79.8% / 20.2%
2          | 88           | 70                 | 18              | 79.5% / 20.5%
3          | 160          | 128                | 32              | 80.0% / 20.0%
4          | 138          | 110                | 28              | 79.7% / 20.3%
5          | 156          | 125                | 31              | 80.1% / 19.9%
6          | 67           | 53                 | 14              | 79.1% / 20.9%
7    

create dataset configuration

In [ ]:
import yaml

config = {
    'path': '/content/dataset',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {0: 'chillies', 1: 'citric' , 2: 'sheasmooth' , 3: 'niveafresh' , 4: 'niveapearl' , 5: 'applecider' , 6: 'soysauce' , 7:'minivaseline' , 8:'toss'}
}

with open('data.yaml', 'w') as f:
    yaml.dump(config, f)

Training model

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train with stronger augmentation
model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=10,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    device=0,
    cache=True,

    # -----------------------------
    # AUGMENTATION SETTINGS
    # -----------------------------
    augment=True,

    # Geometric augmentations
    degrees=15,         # rotate images
    translate=0.1,      # shift image
    scale=0.5,          # zoom in/out
    shear=5,            # shear transform
    perspective=0.0005,

    # Flip augmentations
    flipud=0.2,         # vertical flip
    fliplr=0.5,         # horizontal flip

    # Color augmentations
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # Advanced augmentations
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.1,

    # Save augmented examples
    plots=True
)

Ultralytics 8.4.65 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.2, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78fc55fd7710>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

retraining model


In [ ]:
from ultralytics import YOLO


# 1. Load your OWN best weights instead of 'yolov8n.pt'
# Check if your folder is 'train' or 'train2' etc.
model = YOLO('runs/detect/train/weights/best.pt')

model.train(
        data='/content/data.yaml',
        epochs=60,          # 50 more epochs is usually enough for refinement
        imgsz=320,
        batch=16,
        lr0=0.0001,         # 10x smaller learning rate than before
        optimizer='AdamW',
        freeze=0,          # Freeze backbone to focus on spice-specific details
        cos_lr=True,        # Gradually slows down training for a perfect finish
        close_mosaic=15,    # Clean up detection logic for the final 15 epochs
        device=0
)


Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, pat

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x781799602d80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

export model

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-2/weights/best.pt")

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train-2/weights/best.pt")
model.export(format="ncnn", imgsz=320, half=True)

Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ NCNN export does not support end2end models, disabling end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/train-2/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 13, 2100) (5.9 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 278ms
Prepared 1 package in 265ms
Installed 1 package in 2ms
 + ncnn==1.0.20260526

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx'] not found, attempting AutoUpdate...
Using Python 3.12

'/content/runs/detect/train-2/weights/best_ncnn_model'